In [ ]:
from vpython import canvas, wtext, button, slider, sphere, vector, color, rate
import random

# Флаги и глобальные переменные управления
is_running = True
grid_width = 9   
grid_height = 9  
spacing = 1.3 

# НАСТРОЙКИ ХИМИЧЕСКИХ РЕЖИМОВ (По умолчанию: Очистка воздуха)
# radius - размер молекулы, speed - базовая скорость, name - название для табло
gas_A = {"radius": 0.2, "speed": 3.0, "color": color.cyan, "name": "Азот (N2)"}
gas_B = {"radius": 0.6, "speed": 1.5, "color": color.red,  "name": "Сажа / Дым"}
current_mode_name = "Очистка воздуха (Дым от Азота)"

# Стеки истории (Undo/Redo)
undo_stack = []  
redo_stack = []  

# 1. ФУНКЦИИ ПЕРЕКЛЮЧЕНИЯ ХИМИЧЕСКИХ РЕЖИМОВ
def set_mode_air(b):
    global gas_A, gas_B, current_mode_name
    gas_A = {"radius": 0.2, "speed": 3.0, "color": color.cyan, "name": "Азот (N2)"}
    gas_B = {"radius": 0.6, "speed": 1.5, "color": color.red,  "name": "Сажа / Дым"}
    current_mode_name = "Очистка воздуха (Дым от Азота)"
    reset_statistics()

def set_mode_hydrogen(b):
    global gas_A, gas_B, current_mode_name
    gas_A = {"radius": 0.15, "speed": 4.0, "color": color.yellow, "name": "Водород (H2)"}
    gas_B = {"radius": 0.40, "speed": 2.2, "color": color.orange, "name": "Метан (CH4)"}
    current_mode_name = "Выделение Водорода (H2 из CH4)"
    reset_statistics()

def set_mode_isotopes(b):
    global gas_A, gas_B, current_mode_name
    gas_A = {"radius": 0.30, "speed": 2.5, "color": color.green, "name": "Легкий Изотоп"}
    gas_B = {"radius": 0.36, "speed": 2.4, "color": color.magenta, "name": "Тяжелый Изотоп"}
    current_mode_name = "Разделение Изотопов (Высокая точность!)"
    reset_statistics()

def reset_statistics():
    global blue_total, blue_passed, red_total, red_blocked, all_gas_molecules
    for gas in all_gas_molecules:
        gas.visible = False
    all_gas_molecules.clear()
    blue_total = blue_passed = red_total = red_blocked = 0
    print(f"Режим изменен на: {current_mode_name}. Статистика сброшена.")

# Обычные функции интерфейса
def toggle_simulation(b):
    global is_running
    is_running = not is_running
    if is_running: b.text = "⏸️ Поставить на ПАУЗУ и РЕДАКТИРОВАТЬ"
    else: b.text = "▶️ ЗАПУСТИТЬ симуляцию потока"

def undo_action(b):
    if not is_running and len(undo_stack) > 0:
        last_pos = undo_stack.pop()
        redo_stack.append(last_pos)
        restored_atom = sphere(canvas=sim_canvas, pos=last_pos, radius=0.5, color=color.gray(0.6))
        filter_atoms.append(restored_atom)

def redo_action(b):
    if not is_running and len(redo_stack) > 0:
        target_pos = redo_stack.pop()
        undo_stack.append(target_pos)
        for atom in filter_atoms:
            if (atom.pos - target_pos).mag < 0.01:
                atom.visible = False
                filter_atoms.remove(atom)
                break

def click_action(evt):
    if not is_running:
        clicked_object = sim_canvas.mouse.pick
        if clicked_object and clicked_object in filter_atoms:
            undo_stack.append(vector(clicked_object.pos))
            redo_stack.clear()
            clicked_object.visible = False
            filter_atoms.remove(clicked_object)

def change_width(s):
    global grid_width; grid_width = int(s.value); apply_custom_size()

def change_height(s):
    global grid_height; grid_height = int(s.value); apply_custom_size()

def apply_custom_size():
    global blue_total, blue_passed, red_total, red_blocked, all_gas_molecules, half_w, half_h
    for atom in filter_atoms: atom.visible = False
    filter_atoms.clear()
    for gas in all_gas_molecules: gas.visible = False
    all_gas_molecules.clear()
    undo_stack.clear(); redo_stack.clear()
    blue_total = blue_passed = red_total = red_blocked = 0
    half_w = grid_width / 2; half_h = grid_height / 2
    for x in range(0, 1):
        for y in range(0, grid_height):
            for z in range(0, grid_width):
                pos_vector = vector(x, (y - half_h + 0.5) * spacing, (z - half_w + 0.5) * spacing)
                atom = sphere(canvas=sim_canvas, pos=pos_vector, radius=0.5, color=color.gray(0.6))
                filter_atoms.append(atom)

# 2. СОЗДАНИЕ ГРАФИКИ И ИНТЕРФЕЙСА
sim_canvas = canvas(title="<h3>MolFlow v0.9 — Финальный химический симулятор</h3>", 
                    width=700, height=400, x=0, y=0,
                    center=vector(0, 0, 0), background=color.gray(0.1))

sim_canvas.bind('mousedown', click_action)

print("\n")
control_button = button(bind=toggle_simulation, text="⏸️ Поставить на ПАУЗУ и РЕДАКТИРОВАТЬ")
print("   ")
undo_button = button(bind=undo_action, text="↩️ Undo")
print(" ")
redo_button = button(bind=redo_action, text="↪️ Redo")

print("\n\nПолзунки размеров:")
print("Ширина: ")
width_slider = slider(min=1, max=20, value=9, step=1, bind=change_width)
print("\nВысота: ")
height_slider = slider(min=1, max=20, value=9, step=1, bind=change_height)

# НОВЫЙ БЛОК: КНОПКИ ВЫБОРА ХИМИЧЕСКИХ ВЕЩЕСТВ
print("\n\n🧪 Выберите смесь химических газов для фильтрации:")
btn_mode1 = button(bind=set_mode_air, text="💨 Воздушный фильтр (Дым)")
print("  ")
btn_mode2 = button(bind=set_mode_hydrogen, text="🔋 Выделение Водорода (H2)")
print("  ")
btn_mode3 = button(bind=set_mode_isotopes, text="⚛️ Разделение Изотопов")

print("\n\n")
stats_panel = wtext(text="Инициализация...")

# Стартовый запуск сетки
filter_atoms = []
half_w = grid_width / 2; half_h = grid_height / 2
for x in range(0, 1):
    for y in range(0, grid_height):
        for z in range(0, grid_width):
            pos_vector = vector(x, (y - half_h + 0.5) * spacing, (z - half_w + 0.5) * spacing)
            atom = sphere(canvas=sim_canvas, pos=pos_vector, radius=0.5, color=color.gray(0.6))
            filter_atoms.append(atom)

blue_total = blue_passed = red_total = red_blocked = 0
all_gas_molecules = []
dt = 0.02
frame_counter = 0

# 3. ГЛАВНЫЙ ЦИКЛ СИМУЛЯЦИИ
while True:
    rate(60)
    
    if is_running:
        frame_counter += 1
        
        if frame_counter % 15 == 0:
            spawn_limit_y = half_h * spacing
            spawn_limit_z = half_w * spacing
            start_pos = vector(-8, random.uniform(-spawn_limit_y, spawn_limit_y), random.uniform(-spawn_limit_z, spawn_limit_z))
            
            if random.random() > 0.5:
                # Спавним газ А (Полезный) с динамическими настройками из выбранного режима
                new_gas = sphere(canvas=sim_canvas, pos=start_pos, radius=gas_A["radius"], color=gas_A["color"])
                new_gas.velocity = vector(random.uniform(gas_A["speed"]*0.8, gas_A["speed"]*1.2), random.uniform(-0.2, 0.2), random.uniform(-0.2, 0.2))
                new_gas.type = "blue"
                blue_total += 1
            else:
                # Спавним газ B (Загрязнение)
                new_gas = sphere(canvas=sim_canvas, pos=start_pos, radius=gas_B["radius"], color=gas_B["color"])
                new_gas.velocity = vector(random.uniform(gas_B["speed"]*0.8, gas_B["speed"]*1.2), random.uniform(-0.1, 0.1), random.uniform(-0.1, 0.1))
                new_gas.type = "red"
                red_total += 1
            all_gas_molecules.append(new_gas)
            
        for gas in all_gas_molecules[:]:
            gas.pos = gas.pos + gas.velocity * dt
            
            for atom in filter_atoms:
                distance = (gas.pos - atom.pos).mag
                min_dist = gas.radius + atom.radius
                if distance < min_dist:
                    normal = (gas.pos - atom.pos).norm()
                    gas.velocity = gas.velocity - 2 * gas.velocity.dot(normal) * normal
                    gas.pos = atom.pos + normal * min_dist
                    
            if gas.type == "blue" and gas.pos.x > 6:
                blue_passed += 1; gas.visible = False; all_gas_molecules.remove(gas)
            elif gas.type == "red" and gas.pos.x < -9:
                red_blocked += 1; gas.visible = False; all_gas_molecules.remove(gas)
            elif gas.pos.x > 8 or gas.pos.x < -10:
                gas.visible = False; all_gas_molecules.remove(gas)

    # 4. ОБНОВЛЕНИЕ ПАНЕЛИ СТАТИСТИКИ
    efficiency = (red_blocked / red_total * 100) if red_total > 0 else 0
    flow_rate = (blue_passed / blue_total * 100) if blue_total > 0 else 0
    status_text = "<span style='color: green; font-weight: bold;'>ПОТОК АКТИВЕН</span>" if is_running else "<span style='color: orange; font-weight: bold;'>РЕЖИМ РЕДАКТИРОВАНИЯ</span>"
    
    stats_panel.text = (
        f"<div style='border: 1px solid gray; padding: 10px; background-color: #f9f9f9; font-family: monospace; width: 680px;'>"
        f"<h3>Панель управления MolFlow v0.9</h3>"
        f"<b>Текущий химический режим:</b> <span style='color: blue; font-weight: bold;'>{current_mode_name}</span><br>"
        f"<b>Статус системы:</b> {status_text}<br>------------------------------------------------------------------<br>"
        f"<b>ЗАДЕРЖАНО {gas_B['name'].upper()}:</b> {red_blocked} из {red_total}<br>"
        f"🎯 <span style='color: red; font-weight: bold;'>ЭФФЕКТИВНОСТЬ ОЧИСТКИ: {efficiency:.1f}%</span><br>"
        f"------------------------------------------------------------------<br>"
        f"<b>ПРОПУЩЕНО {gas_A['name'].upper()}:</b> {blue_passed} из {blue_total}<br>"
        f"⚡ <span style='color: darkcyan; font-weight: bold;'>ПРОПУСКНАЯ СПОСОБНОСТЬ: {flow_rate:.1f}%</span>"
        f"</div>"
    )


D:\anaconda\Lib\site-packages\vpython\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>



   
 


Ползунки размеров:
Ширина: 

Высота: 


🧪 Выберите смесь химических газов для фильтрации:
  
  



